In [12]:
"""
Calculate TOU consumption weights from RASS-scaled building profiles.
"""

import pandas as pd
import numpy as np
from pathlib import Path

def get_tou_period(hour, month):
    """SDGE TOU periods"""
    is_summer = 6 <= month <= 10
    
    if is_summer:
        if 16 <= hour < 21:
            return 'summer_peak'
        elif 6 <= hour < 16 or 21 <= hour < 22:
            return 'summer_midpeak'
        else:
            return 'summer_offpeak'
    else:
        if 16 <= hour < 21:
            return 'winter_peak'
        elif 6 <= hour < 16 or 21 <= hour < 22:
            return 'winter_midpeak'
        else:
            return 'winter_offpeak'

def calculate_tou_weights(buildings_dir='./Baseline_SDGE'):
    """Calculate TOU weights from all building files in directory."""
    
    print("="*80)
    print("CALCULATING TOU CONSUMPTION WEIGHTS")
    print("="*80)
    
    # Get all parquet files
    building_files = list(Path(buildings_dir).glob('*-0.parquet'))
    print(f"\nFound {len(building_files)} building files")
    
    # Initialize
    tou_totals = {
        'summer_peak': 0.0, 'summer_midpeak': 0.0, 'summer_offpeak': 0.0,
        'winter_peak': 0.0, 'winter_midpeak': 0.0, 'winter_offpeak': 0.0
    }
    
    # Process each building
    for i, filepath in enumerate(building_files):
        try:
            df = pd.read_parquet(filepath)
            
            # Aggregate to hourly
            load_15min = df['out.electricity.total.energy_consumption'].values
            hourly = load_15min.reshape(-1, 4).sum(axis=1)
            
            # Classify and sum
            for hour_idx, kwh in enumerate(hourly):
                month = min((hour_idx // 730) + 1, 12)
                hour = hour_idx % 24
                period = get_tou_period(hour, month)
                tou_totals[period] += kwh
            
            if (i + 1) % 500 == 0:
                print(f"  Processed {i+1}/{len(building_files)} buildings")
                
        except Exception as e:
            print(f"  Error with {filepath.name}: {e}")
            continue
    
    # Calculate weights
    total = sum(tou_totals.values())
    tou_weights = {k: v/total for k, v in tou_totals.items()}
    
    # Display
    print("\n" + "="*80)
    print("TOU CONSUMPTION WEIGHTS")
    print("="*80)
    print(f"\nTotal consumption: {total:,.0f} kWh")
    for period, weight in tou_weights.items():
        print(f"  {period:20s}: {weight:.4f} ({weight*100:.2f}%)")
    
    return tou_weights

# Run and save
tou_weights = calculate_tou_weights()
df = pd.DataFrame(list(tou_weights.items()), columns=['period', 'weight'])
df.to_csv('tou_weights_sdge.csv', index=False)
print("\n" + "="*80)
print("Saved to: tou_weights_sdge.csv")
print("="*80)

CALCULATING TOU CONSUMPTION WEIGHTS

Found 4317 building files
  Processed 500/4317 buildings
  Processed 1000/4317 buildings
  Processed 1500/4317 buildings
  Processed 2000/4317 buildings
  Processed 2500/4317 buildings
  Processed 3000/4317 buildings
  Processed 3500/4317 buildings
  Processed 4000/4317 buildings

TOU CONSUMPTION WEIGHTS

Total consumption: 32,568,456 kWh
  summer_peak         : 0.1438 (14.38%)
  summer_midpeak      : 0.2064 (20.64%)
  summer_offpeak      : 0.1291 (12.91%)
  winter_peak         : 0.1418 (14.18%)
  winter_midpeak      : 0.2282 (22.82%)
  winter_offpeak      : 0.1506 (15.06%)

Saved to: tou_weights_sdge.csv
